---
title: "Chapter 17: The storytelling capstone"
---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sambaiga/ds-mlops-path/blob/main/tutorials/01-python-basics/17-storytelling-capstone.ipynb) [![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue.svg?logo=jupyter&logoColor=white)](https://raw.githubusercontent.com/sambaiga/ds-mlops-path/main/tutorials/01-python-basics/17-storytelling-capstone.ipynb)

The department head has ten minutes before her next meeting. You have three charts and twelve numbers, and a strong opinion about which courses need help. She asks one question: *"So what do I do?"* Your charts are correct. None of them answer that.

Chapters 14 through 16 taught you how to build a chart and how to make it honest. This chapter is about the step before any of that: knowing who the chart is for and what they need to decide, diagnosing why a draft chart is not landing, and recognising the moment you stop exploring and start explaining. It closes the Data Visualization part by putting every judgment call from the last three chapters, and both libraries, to work on one real case, start to finish. Chapter 18 picks up the engineering practices, uv, ruff, git, testing, that make analysis like this reproducible.

> Callout markers: [book cover page](../../index.qmd#callout-guide).

::: {.callout-note collapse="true" icon=false}
## Learning objectives

By the end of Chapter 17 you will be able to:

| # | Skill | Covered in |
|---|---|---|
| 1 | Name a chart's audience, decision, and medium, and let medium decide the library | Sec. 1 |
| 2 | Diagnose a failing chart with the Trifecta Checkup: content, story, or visual form | Sec. 2 |
| 3 | Decide explore vs explain and change how you build a chart accordingly | Sec. 3 |
| 4 | Rebuild one explanatory chart in matplotlib and lets-plot for two different mediums | Sec. 4 |
| 5 | Produce one explanatory chart, backed by a one-paragraph brief, from a real dataset | Capstone |
:::

## 1. Audience, decision, medium

Every chart decision so far assumed the audience was you: fast defaults, all the data, nothing hidden. The moment someone else needs to act on what you found, three questions come before any chart type: who is this for, what do they need to decide, and how will they see it? Skip these and even a perfectly honest, perfectly decluttered chart can still miss.

In [ ]:
import pandas as pd

df = pd.read_csv("data/university_analytics.csv")

# Same three-course, three-Fall-cohort slice as Chapters 14 to 16: the
# department head's question is about these students specifically.
courses = ["Machine Learning", "Data Structures", "Python Programming"]
semesters = ["Fall 2022", "Fall 2023", "Fall 2024"]
results = df[df["course"].isin(courses) & df["semester"].isin(semesters)].copy()
results.head()

<div class='ark-concept'>
<span class='ark-concept-title'><i class="bi bi-key-fill"></i> Key Concept: Audience, decision, medium</span><br><br>
State three things before opening a plotting library: <b>who</b> will read this chart, <b>what decision</b> they need to make after seeing it, and <b>how</b> they will see it (a live presentation, a slide they read alone, a printed appendix). The same numbers, shown to a department head deciding whether to fund a study-hall program, need a different chart than the same numbers shown to a student checking their own grade.
</div>

Medium has a second consequence beyond how the chart is framed: it decides which library you reach for. matplotlib (Chapter 14) renders to a static image, the right choice for a printed report or a slide someone reads without you in the room. lets-plot (Chapter 15) renders to interactive HTML by default, the right choice for a live meeting or a notebook the reader scrolls through and hovers over themselves. Neither library is more correct: the medium decides, the same way it decided in Chapter 16, Section 5.

<div class='ark-activity'>
<span class='ark-activity-title'><i class="bi bi-puzzle-fill"></i> Activity 1 - Name the audience before you plot</span><br><br>
The department head is deciding whether to fund a mandatory study-hall program next term.<br><br>1. Fill in a dict: <code>brief = {"audience": ..., "decision": ..., "medium": ...}</code>.<br>2. <code>audience</code>: who reads this chart.<br>3. <code>decision</code>: the one thing they do differently depending on what the chart shows.<br>4. <code>medium</code>: how they will see it (live meeting, slide, printed report).<br><br><b>Expected:</b> three short, specific answers, not "everyone" or "management".
</div>

In [ ]:
# TODO: fill in the brief
brief = {"audience": ..., "decision": ..., "medium": ...}
brief

## 2. The Trifecta Checkup

Here is a first attempt at answering the department head's question. Something about it is not working, but "make it prettier" is rarely the actual fix:

In [ ]:
import matplotlib.pyplot as plt

course_scores = results.groupby("course")["final_score"].mean()

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(course_scores.index, course_scores.values, color="#4477AA")
ax.set_ylabel("Mean final score")
ax.set_title("Final score by course")
fig.tight_layout()

<div class='ark-concept'>
<span class='ark-concept-title'><i class="bi bi-key-fill"></i> Key Concept: The Trifecta Checkup</span><br><br>
Berinato's Trifecta Checkup diagnoses a failing chart with three questions, asked in order: (1) <b>Content</b>: is the underlying analysis right, is this even the correct number to show? (2) <b>Story</b>: is there one clear message, could a reader state it in one sentence? (3) <b>Visual form</b>: is the chart type and design right for that message? Fixing (3) before checking (1) is polishing the wrong number. The chart above actually has real content: Machine Learning students do score noticeably lower than Python Programming students. It still fails the content check, for a subtler reason: "course" is not something the department head can act on. She is deciding whether to fund a study-hall program, an intervention aimed at how students study, not at which courses exist. Correct content and useful content are two different questions.
</div>

Fixed for content: not because the course numbers were wrong, but because they were not actionable. Group students by how much they studied instead of by which course they took, since study hours is the variable a study-hall program can actually change:

In [ ]:
results["study_bucket"] = pd.qcut(results["study_hours"], 3, labels=["Low", "Mid", "High"])
score_by_bucket = results.groupby("study_bucket", observed=True)["final_score"].mean()

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(score_by_bucket.index, score_by_bucket.values, color="#4477AA")
ax.set_ylabel("Mean final score")
ax.set_ylim(0, 100)
ax.set_title("Final score by study-hours group")
fig.tight_layout()

Content is fixed. Visual form has its own, independent failure mode: the same correct numbers, drawn dishonestly. Here is the same chart with nothing changed except the y-axis limits:

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(score_by_bucket.index, score_by_bucket.values, color="#4477AA")
ax.set_ylabel("Mean final score")
ax.set_ylim(50, 68)
ax.set_title("Final score by study-hours group")
fig.tight_layout()

<div class='ark-mistake'>
<span class='ark-mistake-title'><i class="bi bi-bug-fill"></i> Common Mistake: A visual-form problem can hide behind correct content</span><br><br>
Both charts above plot the exact same three numbers. The second one starts its y-axis at 50, the same truncation lie from Chapter 16, and it makes the Low-to-High gap look roughly three times larger than it is. Content and visual form are independent checks: this chart passes (1) and still fails (3). Run all three questions every time, even when an earlier one already passed.
</div>

Content and visual form both check out on the honest, zero-baseline version. Now the story check: does this chart state one message, or does the reader have to find it? Right now, nothing points at the answer: the department head still has to read three bar heights and work out the gap herself.

<div class='ark-activity'>
<span class='ark-activity-title'><i class="bi bi-puzzle-fill"></i> Activity 2 - Run the Trifecta Checkup</span><br><br>
Diagnose the truncated-y-axis chart from Chapter 16's capstone using the three questions.<br><br>1. Content: is the number plotted the right one for the question being asked?<br>2. Story: could you state the message in one sentence?<br>3. Visual form: is the chart type and axis honest?<br><br>Write one short sentence answering each question.<br><br><b>Expected:</b> that chart's problem is visual form (the y-axis), not content or story: the checkup should say so explicitly, in order.
</div>

In [ ]:
# TODO: one sentence per question
content_check = "..."
story_check = "..."
visual_form_check = "..."

## 3. Explore vs explain

Every chart in Chapters 14 to 16 so far has been exploratory: fast, all the data, default styling, built to help you understand. The department head does not need to explore. She needs the one message, explained.

<div class='ark-concept'>
<span class='ark-concept-title'><i class="bi bi-key-fill"></i> Key Concept: Explore, then explain</span><br><br>
An <b>exploratory</b> chart is for you: fast, shows everything, default styling, built to find a pattern. An <b>explanatory</b> chart is for someone else: filtered to the one message, branded, annotated, built to make a decision easy. The transition point is the moment you can write the "so what" in one sentence. Until you can write that sentence, you are still exploring, no matter how polished the chart looks.
</div>

The study-hours chart passed content and visual form. What is missing is the explanatory layer: the branding from Chapter 16 and an annotation that states the message, not just shows it. The department head asked for a printed handout, so matplotlib is the right tool here:

In [ ]:
from ark.plot.matplot_theme import configure_matplotlib_style

configure_matplotlib_style(font_size=10, fig_width=5)

gap = score_by_bucket["High"] - score_by_bucket["Low"]

fig, ax = plt.subplots(figsize=(6, 3.5))
colors = ["#DC2626" if b == "High" else "#ADB5BD" for b in score_by_bucket.index]
ax.bar(score_by_bucket.index, score_by_bucket.values, color=colors)
ax.set_ylabel("Mean final score")
ax.set_title(f"Students who study most score {gap:.0f} points higher")
for i, v in enumerate(score_by_bucket.values):
    ax.text(i, v + 1, f"{v:.0f}", ha="center")
fig.tight_layout()

<div class='ark-activity'>
<span class='ark-activity-title'><i class="bi bi-puzzle-fill"></i> Activity 3 - Write the "so what"</span><br><br>
Write the one-sentence takeaway the chart above is making explicit.<br><br>1. State the point gap between the High and Low study-hours groups.<br>2. Say what the department head should do next, in the same sentence.<br><br><b>Expected:</b> one sentence a reader could act on without looking at the chart at all.
</div>

In [ ]:
so_what = "..."
so_what

## 4. One message, two tools

The chart above is right for a printed handout. Now the department head says she would rather review it live, in the meeting, hovering over each bar to see the exact numbers. Same audience, same decision, same message, a different medium, so a different library: lets-plot instead of matplotlib.

In [ ]:
# | include: false
# Each chart also keeps a static copy, which the print edition shows in place of the interactive one.
from ark.plot.static import enable

enable()

In [ ]:
from lets_plot import LetsPlot, aes, geom_bar, ggplot, ggsize, labs, scale_fill_manual

from ark.plot.theme import modern_theme

LetsPlot.setup_html()

score_by_bucket_df = score_by_bucket.reset_index()

(
    ggplot(score_by_bucket_df, aes(x="study_bucket", y="final_score", fill="study_bucket"))
    + geom_bar(stat="identity")
    + scale_fill_manual(values=["#ADB5BD", "#ADB5BD", "#DC2626"])
    + labs(
        title=f"Students who study most score {gap:.0f} points higher",
        x="",
        y="Mean final score",
    )
    + modern_theme()
    + ggsize(500, 320)
)

<div class='ark-concept'>
<span class='ark-concept-title'><i class="bi bi-key-fill"></i> Key Concept: The judgment transfers, the tool does not have to match</span><br><br>
Every decision that made the matplotlib chart explanatory, one message, one colour reserved for emphasis, a title that states the finding, is unchanged here. Only the <code>+</code> layers changed. This is the payoff of Chapter 15's grammar: <code>scale_fill_manual()</code> and <code>labs()</code> work the same way regardless of which chart they are attached to, so switching libraries for a medium change costs a rewrite of the drawing code, not a rethink of the message.
</div>

## Capstone: the department head's chart

Everything from this chapter, applied once, on one real question: does study time predict performance strongly enough to justify funding a study-hall program, and does attendance tell the same story?

<div class='ark-activity'>
<span class='ark-activity-title'><i class="bi bi-puzzle-fill"></i> Capstone exercise - The full judgment loop</span><br><br>
Produce one explanatory chart and a written brief, start to finish:
<ol>
<li><b>Audience, decision, medium:</b> reuse the <code>brief</code> dict from Activity 1. Decide which medium fits: a live meeting or a printed handout.</li>
<li><b>Content:</b> bucket <code>attendance_pct</code> into three groups the same way as <code>study_bucket</code>, and compute mean <code>final_score</code> per group. Is the gap as large as the study-hours gap?</li>
<li><b>Story:</b> write the one-sentence "so what" before touching a plotting library.</li>
<li><b>Visual form:</b> build one chart, branded, that makes the message in step 3 the only thing a reader can take away. Use <code>configure_matplotlib_style()</code> for a printed medium, or lets-plot with <code>modern_theme()</code> for a live one, matching the medium from step 1.</li>
<li><b>Caption:</b> print the brief and the "so what" sentence beneath the chart.</li>
</ol>
<b>Hint:</b> <code>pd.qcut(results["attendance_pct"], 3, labels=["Low", "Mid", "High"])</code> mirrors the <code>study_bucket</code> code from Section 2.
</div>

In [ ]:
# TODO: audience/decision/medium brief (reuse Activity 1)
brief = {"audience": ..., "decision": ..., "medium": ...}

# TODO: content check: mean final_score by attendance bucket
score_by_attendance = ...

# TODO: story: the "so what" sentence
so_what = "..."

# TODO: visual form: one branded, annotated chart (matplotlib or lets-plot, matching brief["medium"])
...

print(brief)
print(so_what)

## Further reading

| Resource | Why it matters |
|---|---|
| Knaflic, C.N. (2015). *Storytelling with Data*. Wiley. | Chapter 1 makes the audience-first case Section 1 is built on |
| Berinato, S. (2016). *Good Charts*. Harvard Business Review Press. | The Trifecta Checkup, in full, alongside the "visual periodic table" |
| Dykes, B. (2019). *Effective Data Storytelling*. Wiley. | The explore/explain distinction and the Data + Narrative + Visuals framework Section 3 draws on |
| Wilkinson, L. (2005). *The Grammar of Graphics*, 2nd ed. Springer. | The formal grammar that makes Section 4's judgment-transfers-across-tools argument true |

## Summary

| Concept | Key rule |
|---|---|
| Audience, decision, medium | State all three before choosing a chart type |
| Medium picks the library | Static and printable: matplotlib. Interactive and live: lets-plot |
| The Trifecta Checkup | Diagnose in order: content, then story, then visual form: each is an independent check |
| Explore vs explain | Exploratory is for you; explanatory is for someone else, filtered to one message |
| The "so what" | Write the one-sentence takeaway before building the explanatory chart |
| One message, two tools | The judgment (message, emphasis, title) transfers across libraries; only the drawing code changes |

**Next:** `18-vscode-setup.qmd`, setting up the engineering environment: editor, interpreter, and linting, that the rest of the book runs inside.